In [1]:
import torch 
import sys
from torch_geometric.data import Data
from datasets.graph_datasets.graph_data_utils import generate_discretised_graph
sys.path.append('..')

/Users/louisgodtfredsen/Desktop/Coding Projects/ML-for-PDEs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path ='../data/test_data/heat_equation_m64_h0_minmax_N200.pt'
input_data = torch.load(data_path)

In [3]:
input_data['X'][0,0].shape

torch.Size([64, 64])

In [4]:
H = 64
x_y_units = torch.tensor([(1/H)*x for x in range(H)])
pos_tensor = torch.cartesian_prod(x_y_units, x_y_units)

In [5]:
pos_tensor

tensor([[0.0000, 0.0000],
        [0.0000, 0.0156],
        [0.0000, 0.0312],
        ...,
        [0.9844, 0.9531],
        [0.9844, 0.9688],
        [0.9844, 0.9844]])

In [149]:
edge_idx, edge_disp, node_spatial_indices = generate_discretised_graph(input_data['X'][0,0], 0.025, 'periodic')

print(edge_idx.shape)
print(edge_disp.shape)
print(node_spatial_indices.shape)

torch.Size([2, 32768])
torch.Size([32768, 2])
torch.Size([4096, 2])


In [176]:
def get_subsample_graph_indices():
    sub_graph_node_indices = []
    num_grpahs = 5
    for _ in range(num_grpahs):
        sub_graph_size = int(torch.randint(1, 4000, (1,1)).squeeze())
        random_node_subsample_idx = torch.randperm(sub_graph_size) # Random Graph node indices
        sub_graph_node_indices.append(random_node_subsample_idx.to(torch.long).squeeze())

    return sub_graph_node_indices

In [178]:
node_indices = get_subsample_graph_indices()[0]
node_indices.shape

torch.Size([1614])

In [179]:
mask = torch.isin(edge_idx[0, :], node_indices)

In [180]:
mask.shape

torch.Size([32768])

In [181]:
# Get edge displacements features for subgraph
mask = torch.isin(edge_idx[0, :], node_indices)
subgraph_node_edges = edge_disp[mask, :]

# Get spatial measurement features for node i.e. measurement for Node A at (x,y)
node_spatial_locs = node_spatial_indices[node_indices, :]
node_spatial_measurements = input_data['X'][0,0][node_spatial_locs[:, 0], node_spatial_locs[:, 1]]

In [187]:
node_indices

tensor([ 999,  190,  349,  ...,  561,  282, 1165])

In [186]:
node_spatial_indices

tensor([[ 0,  0],
        [ 0,  1],
        [ 0,  2],
        ...,
        [63, 61],
        [63, 62],
        [63, 63]])

In [183]:
subgraph_node_edges

tensor([[ 0.0000, -0.0156],
        [ 0.0000,  0.0156],
        [-0.0156,  0.0000],
        ...,
        [-0.0156,  0.0156],
        [-0.0156,  0.0000],
        [-0.0156, -0.0156]])

In [184]:
edge_disp.shape

torch.Size([32768, 2])